# Practice 101 — Instrumental Variables & 2SLS

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import simulate_iv_linear, simulate_iv_binary
from src.plotting import first_stage_fit_plot, weak_instrument_plot, estimate_comparison_plot

## Phase 1 — Endogeneity, and two-stage 2SLS

We simulate data with a **known** true `beta`, where an unobserved confounder
moves both the endogenous regressor `D` and the outcome `y` — so plain OLS of
`y` on `D` is biased. An instrument `Z` (correlated with `D`, but excluded from
the outcome equation) is what fixes this.

In [ ]:
data = simulate_iv_linear(n=500, pi=1.0, seed=0)
print(f"True beta: {data.beta_true}")
data.as_frame().head()

In [ ]:
X_ols = np.column_stack([data.exog, data.endog])
beta_ols_naive = np.linalg.lstsq(X_ols, data.y, rcond=None)[0][-1]
print(f"True beta:  {data.beta_true:.3f}")
print(f"Naive OLS:  {beta_ols_naive:.3f}  (biased — D is correlated with an unobserved confounder)")

### Exercise — `src/_01_tsls_two_stage.py :: tsls_two_stage`

Open `src/_01_tsls_two_stage.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_tsls_two_stage import compare_ols_vs_2sls, tsls_two_stage

compare_ols_vs_2sls(data)
fit_two_stage = tsls_two_stage(data.exog, data.endog, data.instruments, data.y)

## Phase 2 — 2SLS as a single projection

Phase 1's two regressions are algebraically identical to a single projection
of all regressors onto the instrument space. This phase implements that single
formula and validates it against Phase 1's estimator *and* against
`linearmodels.IV2SLS`.

### Exercise — `src/_02_tsls_projection.py :: tsls_projection`

Open `src/_02_tsls_projection.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_tsls_projection import compare_to_linearmodels, tsls_projection

compare_to_linearmodels(data)
fit_projection = tsls_projection(data.exog, data.endog, data.instruments, data.y)

In [ ]:
fig = first_stage_fit_plot(data.instruments[:, 0], data.endog, fit_two_stage.d_hat)
fig

## Phase 3 — First-stage strength & the weak-instrument problem

Relevance (`Cov(Z, D) != 0`) is the one IV condition we can actually check in
the data. The first-stage F statistic tests it; a small F is the classic
"weak instrument" warning sign.

### Exercise — `src/_03_first_stage_strength.py :: first_stage_f_stat`

Open `src/_03_first_stage_strength.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_first_stage_strength import first_stage_f_stat, weak_instrument_simulation

weak = simulate_iv_linear(n=300, pi=0.05, seed=0)
f_strong = first_stage_f_stat(data.exog, data.endog, data.instruments)
f_weak = first_stage_f_stat(weak.exog, weak.endog, weak.instruments)
print(f"First-stage F (pi=1.00, strong instrument): {f_strong:.2f}")
print(f"First-stage F (pi=0.05, weak instrument):   {f_weak:.2f}")

Now sweep the instrument's strength from very weak to strong and, at each
point, simulate many datasets: track 2SLS's bias, OLS's bias, and 95% CI
coverage against the average first-stage F. Watch 2SLS's bias converge toward
OLS's as the instrument weakens, and coverage fall well below the nominal 95%.

In [ ]:
pi_grid = np.geomspace(0.02, 1.5, 12)
results = weak_instrument_simulation(pi_grid, n=300, n_sims=200, seed=0)
fig = weak_instrument_plot(
    results["avg_f"],
    {"2SLS": results["bias_2sls"], "OLS": results["bias_ols"]},
    results["coverage"],
)
fig

## Phase 4 — LATE vs. ATE: the Wald estimator

We switch to a binary encouragement design with three complier types (never-,
always-takers, and compliers), a **known** LATE and ATE, and monotonicity (no
defiers) — see `src/datasets.py :: simulate_iv_binary`. Always-takers benefit
the most from treatment, but that effect can never be revealed by an
instrument that only moves compliers.

In [ ]:
binary_data = simulate_iv_binary(n=4000, complier_share=0.4, seed=0)
print(binary_data.as_frame()["type"].value_counts(normalize=True))
binary_data.as_frame().head()

### Exercise — `src/_04_late_wald.py :: wald_estimator`

Open `src/_04_late_wald.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._04_late_wald import bootstrap_se, compare_ols_wald_truth, wald_estimator

compare_ols_wald_truth(binary_data)
late_hat = wald_estimator(binary_data.y, binary_data.d, binary_data.z)

The Wald estimator is a special case of 2SLS with a binary instrument. Recast
the binary data as IV arrays and confirm Phase 2's `tsls_projection` lands on
the exact same number.

In [ ]:
exog_b = np.ones((len(binary_data.d), 1))
instruments_b = binary_data.z.reshape(-1, 1).astype(float)
fit_binary_2sls = tsls_projection(exog_b, binary_data.d.astype(float), instruments_b, binary_data.y)
print(f"2SLS (as projection):  {fit_binary_2sls.beta_endog:.4f}")
print(f"Wald estimator:        {late_hat:.4f}")

## Phase 5 — End-to-end: OLS vs. 2SLS/Wald vs. the true ATE and LATE

Same coefficient-plot idea as the classical-vs-robust comparison in practice
097, applied here to a different question: which parameter does each method
actually estimate?

In [ ]:
X_ols_b = np.column_stack([np.ones(len(binary_data.d)), binary_data.d])
beta_ols_b = np.linalg.lstsq(X_ols_b, binary_data.y, rcond=None)[0][-1]
se_ols_b, se_wald_b = bootstrap_se(binary_data, n_boot=300, seed=1)

fig = estimate_comparison_plot(
    ["OLS", "2SLS / Wald"],
    [beta_ols_b, late_hat],
    [se_ols_b, se_wald_b],
    {"true ATE": binary_data.ate_true, "true LATE": binary_data.late_true},
)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert np.allclose(fit_two_stage.beta_endog, fit_projection.beta_endog, atol=1e-8), (
    "two-stage and single-projection 2SLS must agree"
)
assert abs(fit_projection.beta_endog - data.beta_true) < 0.3, "2SLS should recover the true beta closely"
assert abs(beta_ols_naive - data.beta_true) > 0.3, "naive OLS should be visibly biased under endogeneity"
assert abs(late_hat - binary_data.late_true) < 0.5, "Wald estimator should recover the true LATE"
assert abs(late_hat - binary_data.ate_true) > 0.3, "LATE should differ from the population ATE here"
print("OK")